# 02: VLM Concept Extraction (Stage 1)

Runs frozen BiomedCLIP over every image in HAM10000, Derm7pt, Fitzpatrick17k, and DDI and saves a 9-dim concept probability bundle per dataset to `outputs/concept_vectors/`.

In [1]:
import sys, subprocess
print("kernel python:", sys.executable)
print("sys.path[0:3]:", sys.path[:3])
try:
    import open_clip
    print("open_clip OK:", open_clip.__file__)
except ImportError as e:
    print("open_clip NOT FOUND in this kernel:", e)

kernel python: /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/bin/python
sys.path[0:3]: ['/opt/anaconda3/lib/python313.zip', '/opt/anaconda3/lib/python3.13', '/opt/anaconda3/lib/python3.13/lib-dynload']


/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


open_clip OK: /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/open_clip/__init__.py


In [2]:
import os, sys, yaml, logging
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from torch.utils.data import DataLoader

from src.data.datasets import build_dataset, default_collate
from src.data.transforms import biomedclip_eval_transform
from src.models.concept_predictor import (
    BiomedCLIPConceptPredictor, CONCEPT_IDS, save_concept_bundle,
)
from src.utils import get_device, describe_device, dataloader_kwargs, seed_everything

logging.basicConfig(level=logging.INFO)
cfg = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
paths = cfg['paths']
seed_everything(cfg.get('seed', 42))
device = get_device()
print('device:', describe_device(device))

device: mps (Apple arm64)


In [3]:
# Load BiomedCLIP (falls back to CLIP-ViT-B-32 if BiomedCLIP weights unavailable)
predictor = BiomedCLIPConceptPredictor(
    model_hub_id=cfg['vlm']['open_clip_hub_id'],
    device=device,
)
print('Loaded:', predictor.model_hub_id)
print('Concepts:', CONCEPT_IDS)

INFO:src.models.concept_predictor:Loading OpenCLIP backbone: hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
INFO:root:Parsing model identifier. Schema: hf-hub, Identifier: microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
INFO:root:Attempting to load config from HF Hub: microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224/resolve/main/open_clip_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224/9f341de24bfb00180f1b847274256e9b65a3a32e/open_clip_config.json "HTTP/1.1 200 OK"
INFO:root:Loaded model config from HF Hub: microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224/resolve/main/open_clip_model.safetensors "H

Loaded: hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
Concepts: ['pigment_network', 'dots_globules', 'blue_white_veil', 'streaks', 'regression', 'vascular', 'asymmetry', 'border', 'color_var']


In [4]:
# Run extraction per dataset, save as .npz bundles
transform = biomedclip_eval_transform(cfg['vlm']['image_size'])
out_dir = PROJECT_ROOT / paths['concept_vectors_dir']
out_dir.mkdir(parents=True, exist_ok=True)

loader_kwargs = dataloader_kwargs(device=device)

for name, root in paths['datasets'].items():
    root = PROJECT_ROOT / root
    if not root.exists():
        print(f'[skip] {name}: {root} missing')
        continue
    try:
        ds = build_dataset(name, root=root, transform=transform)
    except Exception as e:
        print(f'[skip] {name}: {e}')
        continue
    if len(ds) == 0:
        print(f'[skip] {name}: empty dataset (images missing?)')
        continue
    print(f'{name}: {len(ds)} images')
    loader = DataLoader(
        ds, batch_size=cfg['vlm']['batch_size'],
        shuffle=False, collate_fn=default_collate, **loader_kwargs,
    )
    bundle = predictor.predict_dataset(loader)
    save_concept_bundle(out_dir / f'concepts_{name}.npz', bundle)
    print(f'saved -> {out_dir / f"concepts_{name}.npz"}')

ham10000: 10015 images


Concept extraction: 100%|██████████| 313/313 [02:16<00:00,  2.29it/s]


saved -> /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/outputs/concept_vectors/concepts_ham10000.npz
derm7pt: 1011 images


Concept extraction: 100%|██████████| 32/32 [00:12<00:00,  2.54it/s]


saved -> /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/outputs/concept_vectors/concepts_derm7pt.npz
fitzpatrick17k: 4320 images


Concept extraction: 100%|██████████| 135/135 [00:52<00:00,  2.59it/s]


saved -> /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/outputs/concept_vectors/concepts_fitzpatrick17k.npz
ddi: 656 images


Concept extraction: 100%|██████████| 21/21 [00:10<00:00,  1.95it/s]

saved -> /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/outputs/concept_vectors/concepts_ddi.npz


In [5]:
# Quick sanity: concept AUROC on Derm7pt supervised concepts
import numpy as np
from src.evaluation.metrics import per_concept_auroc
from src.models.concept_predictor import load_concept_bundle

p = out_dir / 'concepts_derm7pt.npz'
if p.exists():
    b = load_concept_bundle(p)
    aur = per_concept_auroc(b['concepts'], b['concept_labels'], CONCEPT_IDS)
    for k, v in aur.items():
        print(f'{k:20s}  AUROC={v:.3f}' if not np.isnan(v) else f'{k:20s}  (no GT)')

pigment_network       AUROC=0.634
dots_globules         AUROC=0.699
blue_white_veil       AUROC=0.775
streaks               AUROC=0.691
regression            (no GT)
vascular              AUROC=0.615
asymmetry             (no GT)
border                (no GT)
color_var             (no GT)
